# 00 · Reproduce *features-across-time* (Belrose et al. 2024) on CIFAR-10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtsang27/DS598-FinalProject/blob/colab/reproduce-fat/colab/00_reproduce_features_across_time.ipynb)

This notebook reproduces the vision result from [EleutherAI/features-across-time](https://github.com/EleutherAI/features-across-time):
a network trained on **real CIFAR-10** is evaluated over training on test sets that keep only part of the real structure.

| eval set | keeps | proposal name |
|---|---|---|
| `real` | everything | real CIFAR |
| `gaussian` | per-class mean + covariance (order ≤ 2) | **GM clone** |
| `independent` | each pixel's own distribution, per class (order 1) | ≈ mean-only |
| `cqn` | real images remapped to another class's pixel distributions | order-1 edit |

Expected pattern: early in training, accuracy on `gaussian` / `independent` rises with `real`; later it falls while `real` keeps rising
(the network starts using higher-order structure).

Part 2 trains a second model **on the GM clone** (`--train-on-fake`), the comparison our proposal's real-vs-GM branches build on.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Setup
Mount Google Drive so checkpoints survive a disconnect (optional — set `USE_DRIVE = False` to keep everything in `/content`).

In [ ]:
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/DS598-FinalProject/runs'
else:
    OUT_ROOT = '/content/runs'
os.makedirs(OUT_ROOT, exist_ok=True)
print('Runs will be saved to', OUT_ROOT)

In [ ]:
%cd /content
# Pinned commit of the example code, so every teammate runs the same version
FAT_COMMIT = 'b32a530380cfa761ebe4b7db87879ef9081fa0f5'
!rm -rf features-across-time DS598-FinalProject
!git clone -q https://github.com/EleutherAI/features-across-time.git
!cd features-across-time && git checkout -q $FAT_COMMIT
!git clone -q -b colab/reproduce-fat https://github.com/jtsang27/DS598-FinalProject.git
# PyPI concept-erasure (0.2.4) lacks QuantileNormalizer, so install the pinned GitHub version
!pip install -q einops datasets transformers accelerate "git+https://github.com/EleutherAI/concept-erasure@9f51753821316a1edacf78b52b464ab26d40e60a"

Patch the example so it runs on one Colab GPU (auto device, no bf16 on T4, no wandb, skip eval sets that need missing files).
See `colab/patch_fat.py` for the exact list.

In [ ]:
!python DS598-FinalProject/colab/patch_fat.py features-across-time

## 2 · Configure
The paper trains for 65,536 steps (≈ 2–3 h on a T4). Start with 4,096 to check everything works, then raise it.
Evaluation happens at steps 1, 2, 4, 8, … (log-spaced), so short runs still give a full curve.

In [ ]:
MAX_STEPS = 4096        # paper: 65536
SEED      = 0
NET       = 'convnext'  # convnext (= ConvNeXt-V2 atto) or swin (= Swin-tiny); bare 'regnet' crashes in the original
DATASET   = 'uoft-cs/cifar10'  # full HF id; plain 'cifar10' fails with newer huggingface_hub
EVAL_SETS = 'real,gaussian,independent,cqn'
WARMUP    = 2000        # original value; with short runs you may want MAX_STEPS // 32

def run_dir(kind):
    return f'{OUT_ROOT}/{DATASET}/{NET}/{kind}-seed{SEED}-steps{MAX_STEPS}'

os.environ.update(FAT_MAX_STEPS=str(MAX_STEPS), FAT_EVAL_SETS=EVAL_SETS, FAT_WARMUP=str(WARMUP))

## 3 · Train on real CIFAR-10

In [ ]:
%cd /content/features-across-time
os.environ['FAT_OUTPUT_DIR'] = run_dir('real')
!python -m scripts.inference.train_vision --datasets $DATASET --nets $NET --seed $SEED

## 4 · Train on the GM clone (optional, same settings)
Every training batch is sampled fresh from the per-class Gaussians; evaluation is still on all eval sets including `real`.

In [ ]:
os.environ['FAT_OUTPUT_DIR'] = run_dir('gm')
!python -m scripts.inference.train_vision --datasets $DATASET --nets $NET --seed $SEED --train-on-fake

## 5 · Plot accuracy across training
Reads the evaluation log saved in the last checkpoint of each run.

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt

def load_history(path):
    ckpts = sorted(glob.glob(f'{path}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
    if not ckpts:
        return None
    with open(f'{ckpts[-1]}/trainer_state.json') as f:
        hist = json.load(f)['log_history']
    rows = [h for h in hist if any(k.endswith('_acc') for k in h)]
    return pd.DataFrame(rows).groupby('step').first().reset_index()

runs = {k: load_history(run_dir(k)) for k in ['real', 'gm']}
runs = {k: v for k, v in runs.items() if v is not None}

fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 4), squeeze=False)
for ax, (kind, df) in zip(axes[0], runs.items()):
    for name in EVAL_SETS.split(','):
        col = f'eval_{name}_acc'
        if col in df:
            ax.plot(df['step'], df[col], marker='o', ms=3, label=name,
                    color='black' if name == 'real' else None)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('training step'); ax.set_ylabel('accuracy')
    ax.set_title(f'trained on {kind.upper()} CIFAR-10 ({NET})')
    ax.axhline(0.1, ls=':', c='gray', lw=1)
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Tables for the write-up
for kind, df in runs.items():
    print(kind); display(df.filter(regex='^step$|_acc$').round(3))